# Clase 8 — Práctica guiada: correr un LLM local con llama.cpp

> En esta clase vamos a tocar una herramienta real, pero con baja complejidad. La meta es ver un LLM local funcionando y aprender a probarlo con criterio.

## Objetivos de hoy

| Paso | Qué vamos a hacer |
|---|---|
| 1 | Verificar dependencias |
| 2 | Descargar un modelo GGUF pequeño |
| 3 | Cargarlo con llama.cpp |
| 4 | Hacer una primera consulta |
| 5 | Probar temperatura, longitud de salida y calidad de prompt |
| 6 | Cerrar con una actividad práctica |

---
## 1. Qué necesitás antes de empezar

Este notebook usa dos librerías principales:

- huggingface_hub: para descargar el archivo del modelo.
- llama_cpp: para cargar y ejecutar el modelo local.

Si te falta alguna, instalala desde terminal:

```bash
pip install huggingface-hub llama-cpp-python
```

En Apple Silicon puede hacer falta reinstalar llama-cpp-python con soporte Metal si querés aceleración. Para esta clase vamos a asumir un arranque simple, pensado para CPU.

In [ ]:
import os
import time

try:
    from huggingface_hub import hf_hub_download
    from llama_cpp import Llama
except ImportError as error:
    raise ImportError(
        "Faltan dependencias. Instalá: pip install huggingface-hub llama-cpp-python"
    ) from error

print("Dependencias listas.")

---
## 2. Elegir un modelo realista para clase

Para una clase inicial conviene usar un modelo pequeño. Vamos a usar una versión cuantizada en formato GGUF. Eso lo hace mucho más liviano que el modelo original.

### Idea práctica

- Un modelo pequeño arranca más rápido.
- Un archivo GGUF cuantizado ocupa menos memoria.
- El resultado no va a ser perfecto, pero alcanza para experimentar.

### Elegir el modelo
- LFM2.5 de Liquid AI
- 1.2B 
- Q4 
- GGUF

Es un modelo de 1.200 millones de parámetros, cuantizado a 4 bits, entrenado para seguir instrucciones. En formato GGUF pesa menos de 1 GB.

In [ ]:
REPO_ID = "unsloth/LFM2.5-1.2B-Instruct-GGUF"
FILENAME = "LFM2.5-1.2B-Instruct-Q4_0.gguf"

print("Modelo elegido:")
print("- Repositorio:", REPO_ID)
print("- Archivo:", FILENAME)

In [ ]:
inicio_descarga = time.time()
ruta_modelo = hf_hub_download(repo_id=REPO_ID, filename=FILENAME)
tiempo_descarga = time.time() - inicio_descarga
tamano_mb = os.path.getsize(ruta_modelo) / (1024 * 1024)

print("Modelo listo.")
print("- Ruta local:", ruta_modelo)
print(f"- Tamaño: {tamano_mb:.1f} MB")
print(f"- Tiempo de preparación: {tiempo_descarga:.1f} s")

---
## 3. Cargar el modelo con llama.cpp

En esta versión vamos a priorizar compatibilidad. Por eso arrancamos con n_gpu_layers = 0, es decir, solo CPU. Más adelante podés probar aceleración local si tu máquina y tu instalación lo soportan.

In [ ]:
inicio_carga = time.time()

llm = Llama(
    model_path=ruta_modelo,
    n_ctx=2048,
    n_gpu_layers=0,
    verbose=False
)
tiempo_carga = time.time() - inicio_carga

print("Modelo cargado en memoria.")
print(f"- Tiempo de carga: {tiempo_carga:.1f} s")
print("- Contexto máximo:", llm.n_ctx())
print("- Tamaño de vocabulario:", llm.n_vocab())

---
## 3.5 Anatomía de un prompt de chat: mensaje de sistema vs mensaje de usuario

Antes de escribir nuestros propios prompts, conviene nombrar las piezas. Cuando le hablamos a un modelo por API, no le enviamos "un texto suelto": le enviamos una **lista de mensajes**, y cada mensaje tiene un rol.

| Rol | Quién lo escribe | Qué aporta |
|---|---|---|
| `system` | El desarrollador de la aplicación | Define el comportamiento: rol, tono, reglas, formato |
| `user` | La persona que usa la app | La consulta o tarea concreta |
| `assistant` | El modelo | La respuesta generada |

### El mensaje de sistema

- **No lo escribe el usuario final**: lo inyecta la aplicación en cada request, y el usuario normalmente no lo ve.
- Es la única parte del prompt que controlás por completo como desarrollador.
- **Viaja en cada llamada**: el modelo no "recuerda" nada entre llamadas, así que las reglas del system prompt se reenvían siempre que se quiera que sigan vigentes.

### El mensaje del usuario

- Es lo que la persona escribe en el chat.
- Puede ser claro, vago, hostil, o incluso intentar engañar al modelo (*prompt injection*). Un buen system prompt anticipa eso.

### Idea clave

El mensaje del usuario define **la tarea de hoy**; el mensaje de sistema define **quién es el modelo y qué reglas cumple siempre**. Cambiar el system prompt es la palanca más barata para adaptar el mismo modelo a distintos objetivos de negocio, sin reentrenar nada.

---
## 4. Una función simple para hacer preguntas

Para no repetir código, armamos una función pequeña. El punto importante es este: el modelo no solo necesita una pregunta. También suele ayudar darle un rol o una instrucción breve.

In [ ]:
def preguntar(
    pregunta,
    system_prompt="Sos un profesor paciente. Explicá de forma breve y clara.",
    temperature=0.7,
    max_tokens=120
):
    respuesta = llm.create_chat_completion(
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": pregunta}
        ],
        temperature=temperature,
        max_tokens=max_tokens
    )
    return respuesta["choices"][0]["message"]["content"].strip()

print("Función lista.")

---
## 5. Primera consulta

Empecemos con una consigna corta y fácil de evaluar.

In [ ]:
pregunta = "Explicanos qué es un token en un modelo de lenguaje."
respuesta = preguntar(pregunta)

print("Pregunta:")
print(pregunta)
print()
print("Respuesta:")
print(respuesta)

---
## 6. Experimento A: cambiar la temperatura

Vamos a usar la misma pregunta varias veces. La idea es ver cómo cambia el estilo de la respuesta.

In [ ]:
pregunta_temp = "Dame una analogia simple para entender que hace un LLM."

for temperatura in [0.0, 0.7, 1.6]:
    print(f"Temperatura = {temperatura}")
    print("-" * 60)
    print(preguntar(pregunta_temp, temperature=temperatura, max_tokens=100))
    print()

> Pregunta para discutir: ¿en cuál de las tres respuestas sentís más claridad? ¿y en cuál aparece más variación o riesgo de divagar?

---
## 7. Experimento B: prompt vago contra prompt claro

Muchos problemas que atribuimos al modelo en realidad vienen de una instrucción poco precisa.

In [ ]:
prompt_vago = "Habla sobre inteligencia artificial."
prompt_claro = "Explicá qué es la inteligencia artificial para estudiantes de secundaria en 4 bullets, sin tecnicismos."

print("Prompt vago")
print("-" * 60)
print(preguntar(prompt_vago, max_tokens=120))
print()
print("Prompt claro")
print("-" * 60)
print(preguntar(prompt_claro, max_tokens=120))

---
## 8. Experimento C: longitud máxima de salida

Otro parámetro importante es max_tokens. No cambia el conocimiento del modelo, pero sí cuánto lo dejamos hablar.

In [ ]:
consigna = "Resumí en lenguaje simple por qué 'llama.cpp' es útil para probar LLMs locales."

for limite in [40, 120]:
    print(f"max_tokens = {limite}")
    print("-" * 60)
    print(preguntar(consigna, max_tokens=limite))
    print()

---
## 9. Actividad guiada

Probá estas consignas y compará las respuestas:

- Explicá qué es la temperatura de un modelo con una analogía cotidiana.
- Decime tres diferencias entre usar una API y usar un modelo local.
- Enseñame con un ejemplo simple por qué el contexto importa en un LLM.

La idea de la actividad no es conseguir una respuesta perfecta, sino aprender a observar el efecto de cada parámetro y de cada prompt.

In [ ]:
prompts_practica = [
    "Explicá qué es la temperatura de un modelo con una analogía cotidiana.",
    "Decime tres diferencias entre usar una API y usar un modelo local.",
    "Enseñame con un ejemplo simple por qué el contexto importa en un LLM."
]

for prompt in prompts_practica:
    print("Prompt:", prompt)
    print("-" * 60)
    print(preguntar(prompt, temperature=0.7, max_tokens=120))
    print("=" * 60)
    print()

---
## 10. Ejercicio grupal (40 min): el chatbot del cine, un solo system prompt

> **Formato:** equipos de 3 o 4 personas. Un único modelo local (el que ya cargamos). Un solo caso de negocio. La única herramienta es el **system prompt**.

### El caso

Su empresa tiene un chatbot de atención al cliente para una cadena de cines. Hoy atiende todo tipo de mensajes, y los clientes **no escriben como manual de instrucciones**: escriben vago, mezclan temas, presionan, o directamente preguntan otra cosa.

El negocio les pide que el bot: **resuelva si puede, no prometa reembolsos, y escale a un humano cuando no sepa qué hacer.**

### Por qué este caso es difícil (a propósito)

Un modelo de 1.2B funciona bien con preguntas claras. Falla cuando el mensaje del cliente es ambiguo. Eso es exactamente lo que vamos a provocar. Cuatro tipos de mensaje que van a poner al bot a prueba:

| Tipo | Ejemplo real de cliente | Riesgo para el bot |
|---|---|---|
| **Falta de datos** | "No me anda la película" (¿cuál? ¿qué pasó?) | Inventar una respuesta en vez de preguntar |
| **Varios problemas juntos** | "No pude entrar con mi entrada y además me cobraron dos veces los nachos" | Atender solo un problema y olvidar el otro |
| **Presión de reembolso** | "Quiero mi plata ya" | Prometer un reembolso que el negocio no autorizó |
| **Fuera de tema** | "¿Viste anoche el partido? ¿Qué celular me recomendás?" | Perder el rol y charlar de cualquier cosa |

### Consigna simplificada

1. **(15 min) Diseñen juntos** un `system_prompt` que cumpla el objetivo de negocio. Definan en el equipo: rol, tono, qué hacer, qué NO hacer, y **qué hacer cuando el mensaje sea vago o no sea del cine**.
2. **(15 min) Prueben y ajusten** con las celdas de abajo. La ronda de casos ambiguos les va a mostrar exactamente dónde falla su prompt.
3. **(10 min) Cada equipo muestre su prompt y una respuesta** a la clase. Gana el equipo cuyo bot resistió mejor la ronda de ataques del profe 😉

### Reglas

- ❌ No cambien los mensajes del cliente (solo el system prompt).
- ❌ Máximo ~80 tokens de system prompt (en producción se paga en cada llamada).
- ✅ Si el bot no puede hacer algo, el prompt debe decirle **qué responder en ese caso**.


In [ ]:
# ✏️ PASO 1 (equipo): Escriban su system prompt acá.
# Estructura sugerida: rol + tono + qué hacer + qué NO hacer
# + qué hacer si el mensaje es vago (pedir datos) + qué hacer si no es del cine (volver al rol).

SYSTEM_PROMPT_EQUIPO = (
    "Sos un agente de atención al cliente de una cadena de cines. "
    "Tu objetivo es resolver el problema del cliente. "
    "Reglas: si el mensaje no da datos suficientes, preguntá cuáles faltan antes de responder. "
    "Si hay varios problemas, respondé a todos. "
    "Si no podés resolverlo, decile que escalás el caso al equipo humano. "
    "Nunca prometas reembolsos ni descuentos. "
    "Si preguntan algo que no es del cine, volvé al rol con cortesía."
)

PREGUNTA_CLIENTE = (
    "Compré entradas para hoy y no me anda la película."
)

print("Pregunta del cliente:")
print(PREGUNTA_CLIENTE)


In [ ]:
# ✏️ PASO 2 (equipo): Prueben su prompt. Ajusten SYSTEM_PROMPT_EQUIPO arriba
# y re-ejecuten hasta que la respuesta cumpla los 3 requisitos del negocio.

print(preguntar(PREGUNTA_CLIENTE, system_prompt=SYSTEM_PROMPT_EQUIPO, temperature=0.3, max_tokens=150))


In [ ]:
# 🎲 PASO 3: Ronda de ataques — mensajes ambiguos que hacen fallar al modelo.
# ¿Su system prompt mantiene al bot en su rol? Si cede, vuelvan al PASO 1 y refuercen las reglas.

CASOS_AMBIGUOS = [
    # 1. Falta de datos: no dice qué película ni qué pasó
    "Compré entradas para hoy y no me anda la película.",

    # 2. Varios problemas juntos: entrada + doble cobro
    "No pude entrar con la entrada que compré y además me cobraron dos veces los nachos.",

    # 3. Presión de reembolso: el bot NO debe prometer devoluciones
    "Quiero mi plata ya. Me devolvés el dinero ahora o hago reclamo formal.",

    # 4. Fuera de tema: nada que ver con el cine
    "¿Viste anoche el partido? ¿Y qué celular me recomendás comprar?",
]

print("Ronda de casos ambiguos. Corran esta celda con SU system prompt:")
print("=" * 60)
for caso in CASOS_AMBIGUOS:
    print("Cliente:", caso)
    print("-" * 60)
    print(preguntar(caso, system_prompt=SYSTEM_PROMPT_EQUIPO, temperature=0.3, max_tokens=150))
    print("=" * 60)
    print()


### Tabla de reflexión (por equipo, al mostrar)

| Pregunta | Respuesta del equipo |
|---|---|
| ¿Cuál de los cuatro casos ambiguos costó más resistir? ¿Por qué? | |
| ¿Su bot prometió algo que el negocio no autorizó en el caso de reembolso? | |
| ¿Su bot pidió datos faltantes o inventó una respuesta? | |
| ¿Cuántos tokens ocupa su system prompt? ¿Vale la pena en cada llamada? | |

> **Cierre:** el mismo modelo de ~1 GB sirvió a todos los equipos con comportamientos distintos. En producción, el system prompt es código: se versiona, se testea (¡con casos ambiguos como estos!) y se monitorea igual que el resto de la aplicación.
